# Week 1: Strategic Planning and Data Exploration in Logistics
## Zomato delivery data: audit, performance exploration, and KPIs

**Scenario:** A food-delivery operations team wants to understand delivery-time variation and establish planning baselines. This is descriptive analysis, not causal inference.

**Metric labels:** `time_taken_min` is observed. Group averages, distance, and percentiles are derived. Long delivery uses the 75th percentile as a transparent analytical proxy because promised delivery time is absent. This is not contractual on-time delivery.

Invalid coordinate pairs are excluded from distance analysis. Remaining distances are approximate straight-line distances, not road distances. No regression, clustering, or optimization models are included.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
ROOT=Path.cwd()
if not (ROOT/'data/cleaned_logistics_data.csv').exists(): ROOT=ROOT.parent
DATA=ROOT/'data/cleaned_logistics_data.csv'; FIG=ROOT/'visualizations'; FIG.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(DATA,parse_dates=['order_date'],low_memory=False)
print(f'Loaded {len(df):,} rows x {len(df.columns)} columns')

Loaded 45,584 rows x 33 columns


## 1. Structured data audit
Review schema, missingness, duplicates, category coverage, and numeric distributions. No rows are removed; numeric missing values remain missing.

In [2]:
print('Shape:',df.shape,'| duplicate rows:',int(df.duplicated().sum()),'| duplicate IDs:',int(df.id.duplicated().sum()))
display(df.dtypes.rename('dtype').to_frame())
audit=pd.DataFrame({'missing':df.isna().sum(),'missing_pct':(100*df.isna().mean()).round(2)})
display(audit[audit.missing.gt(0)].sort_values('missing',ascending=False))
for c in ['road_traffic_density','weather_conditions','type_of_vehicle','vehicle_condition','city','festival','day_of_week','has_multiple_deliveries']:
 print('\n'+c); display(df[c].value_counts(dropna=False).rename('records').to_frame())
display(df[['time_taken_min','approx_delivery_distance_km','multiple_deliveries','vehicle_condition','order_hour']].describe().T.round(2))

Shape: (45584, 33) | duplicate rows: 0 | duplicate IDs: 0
                                       dtype
id                                       str
delivery_person_id                       str
delivery_person_age                  float64
delivery_person_ratings              float64
restaurant_latitude                  float64
restaurant_longitude                 float64
delivery_location_latitude           float64
delivery_location_longitude          float64
order_date                    datetime64[us]
time_orderd                              str
time_order_picked                        str
weather_conditions                       str
road_traffic_density                     str
vehicle_condition                      int64
type_of_order                            str
type_of_vehicle                          str
multiple_deliveries                  float64
festival                                 str
city                                     str
time_taken_min                         int

## 2. KPIs and subgroup comparisons
Long-delivery rate counts records strictly above the observed 75th percentile. It is a tail indicator, **not on-time delivery**. Distance includes valid coordinate pairs only.

In [3]:
dt=df.time_taken_min.dropna(); threshold=float(dt.quantile(.75))
kpis=pd.Series({'Average delivery time (min)':dt.mean(),'Median delivery time (min)':dt.median(),'Long-delivery threshold: P75 (min)':threshold,'Long-delivery rate: >P75 (%)':100*(dt>threshold).mean(),'Average approximate distance (km; valid coordinates)':df.approx_delivery_distance_km.mean(),'Records in distance KPI':df.approx_delivery_distance_km.notna().sum()})
display(kpis.to_frame('value').round(2))
def by_mean(col): return df.groupby(col,dropna=False).time_taken_min.agg(['count','mean','median']).sort_values('mean',ascending=False).round(2)
for c in ['road_traffic_density','type_of_vehicle','weather_conditions','vehicle_condition','has_multiple_deliveries','city','festival','day_of_week','order_hour']:
 print('\nAverage delivery time by',c); display(by_mean(c))
print('\nLive-operation KPIs: median/P90 duration; on-time rate vs promise; cancellation/failure rate; cost per completed delivery; courier utilization by zone/hour. Promise, failures, costs, and shifts are absent here.')

                                                         value
Average delivery time (min)                              26.29
Median delivery time (min)                               26.00
Long-delivery threshold: P75 (min)                       32.00
Long-delivery rate: >P75 (%)                             24.71
Average approximate distance (km; valid coordinates)      9.72
Records in distance KPI                               41513.00

Average delivery time by road_traffic_density
                      count   mean  median
road_traffic_density                      
Jam                   14139  31.18    31.0
High                   4423  27.24    27.0
Medium                10945  26.70    27.0
Unknown                 601  26.54    26.0
Low                   15476  21.27    20.0

Average delivery time by type_of_vehicle
                  count   mean  median
type_of_vehicle                       
motorcycle        26429  27.61    26.0
bicycle              68  26.43    26.0
scooter      

## 3. Visual exploration
PNG figures are saved to `visualizations/`. Pillow creates the charts in this environment. Box plots show P10–P90 whiskers and interquartile range.

In [4]:
W,H=1100,600; F=ImageFont.load_default(); BLUE='#2878B5'; NAVY='#17324D'; GRID='#D9E0E7'; INK='#243746'
def canvas(title,label):
 im=Image.new('RGB',(W,H),'white'); d=ImageDraw.Draw(im); d.text((65,20),title,fill=NAVY,font=F); d.text((15,65),label,fill=INK,font=F); return im,d,(90,85,W-35,H-85)
def hist(v,title,xlab,name):
 v=pd.to_numeric(pd.Series(v),errors='coerce').dropna().to_numpy(); im,d,(l,t,r,b)=canvas(title,'Records'); counts,_=np.histogram(v,bins=22); mx=max(counts.max(),1); w=(r-l)/len(counts)
 for k in range(5):
  y=b-k*(b-t)/4; d.line((l,y,r,y),fill=GRID); d.text((48,y-5),str(int(mx*k/4)),fill=INK,font=F)
 for i,n in enumerate(counts): d.rectangle((l+i*w+2,b-n/mx*(b-t),l+(i+1)*w-2,b),fill=BLUE)
 d.text((l,b+12),f'{v.min():.1f}',fill=INK,font=F); d.text((r-35,b+12),f'{v.max():.1f}',fill=INK,font=F); d.text((W//2-80,H-25),xlab,fill=INK,font=F); im.save(FIG/name); print('Saved',name)
def box(col,title,name,order=None):
 z=df[[col,'time_taken_min']].copy(); z[col]=z[col].astype('string').fillna('Unknown'); z=z.dropna(subset=['time_taken_min']); order=order or list(z.groupby(col).time_taken_min.median().sort_values(ascending=False).index); lo=float(z.time_taken_min.min()); hi=float(z.time_taken_min.max()); im,d,(l,t,r,b)=canvas(title,'Delivery time (min)'); ym=lambda v:b-(float(v)-lo)/max(hi-lo,1)*(b-t)
 for k in range(5):
  val=lo+(hi-lo)*k/4; y=ym(val); d.line((l+145,y,r,y),fill=GRID); d.text((l+110,y-5),f'{val:.0f}',fill=INK,font=F)
 slot=(r-l-150)/max(len(order),1)
 for i,label in enumerate(order):
  v=z.loc[z[col].astype(str)==str(label),'time_taken_min']
  if not len(v): continue
  q10,q25,q50,q75,q90=v.quantile([.1,.25,.5,.75,.9]); x=l+150+(i+.5)*slot; bw=min(48,slot*.55); d.line((x,ym(q10),x,ym(q90)),fill=NAVY,width=2); d.rectangle((x-bw/2,ym(q75),x+bw/2,ym(q25)),fill='#DCEAF5',outline=BLUE,width=2); d.line((x-bw/2,ym(q50),x+bw/2,ym(q50)),fill=NAVY,width=3); d.text((x-32,b+10),str(label)[:9]+f' n{len(v)}',fill=INK,font=F)
 im.save(FIG/name); print('Saved',name)
def scatter(name):
 z=df.loc[df.coordinates_valid.eq(True),['approx_delivery_distance_km','time_taken_min']].dropna().sample(min(int(df.coordinates_valid.sum()),5000),random_state=7); xm=z.iloc[:,0].max(); ymx=z.iloc[:,1].max(); im,d,(l,t,r,b)=canvas('Delivery time versus approximate distance','Delivery time (min)'); d.line((l,b,r,b),fill=NAVY,width=2); d.line((l,t,l,b),fill=NAVY,width=2)
 for x0,y0 in z.itertuples(index=False,name=None):
  x=l+x0/max(xm,1)*(r-l); y=b-y0/max(ymx,1)*(b-t); d.ellipse((x-2,y-2,x+2,y+2),fill=BLUE)
 d.text((W//2-110,H-25),'Approximate straight-line distance (km)',fill=INK,font=F); im.save(FIG/name); print('Saved',name,'sample=',len(z))
hist(df.time_taken_min,'Distribution of delivery time','Delivery time (min)','delivery_time_distribution.png')
box('road_traffic_density','Delivery time by traffic density','delivery_time_by_traffic.png')
box('weather_conditions','Delivery time by weather','delivery_time_by_weather.png')
box('type_of_vehicle','Delivery time by vehicle type','delivery_time_by_vehicle.png')
hist(df.loc[df.coordinates_valid.eq(True),'approx_delivery_distance_km'],'Distance: valid coordinates only','Straight-line km','valid_delivery_distance_distribution.png')
scatter('delivery_time_vs_approx_distance.png')
box('day_of_week','Delivery time by day of week','delivery_time_by_day_of_week.png',['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
box('has_multiple_deliveries','Delivery time by multiple-delivery status','delivery_time_by_multiple_delivery_status.png',['True','False','<NA>'])
print('Generated 8 requested figures.')

Saved delivery_time_distribution.png
Saved delivery_time_by_traffic.png
Saved delivery_time_by_weather.png
Saved delivery_time_by_vehicle.png
Saved valid_delivery_distance_distribution.png
Saved delivery_time_vs_approx_distance.png sample= 5000
Saved delivery_time_by_day_of_week.png
Saved delivery_time_by_multiple_delivery_status.png
Generated 8 requested figures.


### Delivery time distribution
The upper tail helps frame operational buffers; P75 is not a customer promise.

![Delivery time](../visualizations/delivery_time_distribution.png)

### Traffic and weather
Category differences can inform planning hypotheses, not causal claims.

![Traffic](../visualizations/delivery_time_by_traffic.png)

![Weather](../visualizations/delivery_time_by_weather.png)

### Vehicle type
Sparse groups, notably bicycles, have unstable estimates.

![Vehicle](../visualizations/delivery_time_by_vehicle.png)

### Distance and duration
Valid coordinates only. Great-circle distance is not road distance. Scatter is a fixed-seed sample for legibility; KPIs use all valid points.

![Distance](../visualizations/valid_delivery_distance_distribution.png)

![Time and distance](../visualizations/delivery_time_vs_approx_distance.png)

### Day of week and multiple deliveries
Weekly comparisons inform staffing hypotheses. Batch composition and sequence are unavailable.

![Weekday](../visualizations/delivery_time_by_day_of_week.png)

![Multi delivery](../visualizations/delivery_time_by_multiple_delivery_status.png)

## 4. EDA findings
These are descriptive associations, not causal effects.

In [5]:
print(f'1. Typical duration: median {dt.median():.1f} min; mean {dt.mean():.1f} min. P75 {threshold:.0f} min; {100*(dt>threshold).mean():.1f}% strictly above (analytical proxy).')
g=df.groupby('road_traffic_density').time_taken_min.mean().dropna(); print(f'2. Traffic means span {g.min():.1f}-{g.max():.1f} min; highest {g.idxmax()} ({g.max():.1f}). Investigate, not causal.')
g=df.groupby('weather_conditions').time_taken_min.mean().dropna(); print(f'3. Weather means span {g.min():.1f}-{g.max():.1f} min ({g.idxmin()} to {g.idxmax()}); categories are broad labels.')
g=df.groupby('type_of_vehicle').time_taken_min.agg(['mean','count']); print('4. Vehicle means min/n:', '; '.join(f'{i} {r["mean"]:.1f}/{int(r["count"])}' for i,r in g.iterrows()),'— sparse categories need caution.')
v=df.approx_delivery_distance_km.dropna(); print(f'5. {len(v):,} rows ({len(v)/len(df):.1%}) have valid coordinates; mean straight-line distance {v.mean():.1f} km and median {v.median():.1f} km.')
n=int((~df.coordinates_valid).sum()); print(f'6. {n:,} rows ({n/len(df):.1%}) have invalid coordinates and are excluded from distance aggregates; improve location validation.')
g=df.groupby('has_multiple_deliveries').time_taken_min.mean(); print('7. Mean time by multiple-delivery flag:',g.round(1).to_dict(),'; batch composition is absent.')
g=df.groupby('day_of_week').time_taken_min.mean().dropna(); print(f'8. Weekday means span {g.min():.1f}-{g.max():.1f} min; verify counts and more seasons before staffing changes.')

1. Typical duration: median 26.0 min; mean 26.3 min. P75 32 min; 24.7% strictly above (analytical proxy).
2. Traffic means span 21.3-31.2 min; highest Jam (31.2). Investigate, not causal.
3. Weather means span 21.9-28.9 min (Sunny to Cloudy); categories are broad labels.
4. Vehicle means min/n: bicycle 26.4/68; electric_scooter 24.5/3814; motorcycle 27.6/26429; scooter 24.5/15273 — sparse categories need caution.
5. 41,513 rows (91.1%) have valid coordinates; mean straight-line distance 9.7 km and median 9.2 km.
6. 4,071 rows (8.9%) have invalid coordinates and are excluded from distance aggregates; improve location validation.
7. Mean time by multiple-delivery flag: {False: 25.5, True: 41.6} ; batch composition is absent.
8. Weekday means span 25.2-27.8 min; verify counts and more seasons before staffing changes.


## 5. Operational KPI set and next data needs
Track in a live operation: (1) median and P90 delivery time by zone/hour, (2) on-time rate against the actual promised timestamp, (3) cancellation/failure rate, (4) cost per completed order, and (5) courier utilization. Several require fields absent here. This 2022 dataset lacks promised timestamps, road routes, courier shifts/capacity, costs, and cancellation outcomes. Collect these before service commitments or live dispatch. No modeling stage is included.